In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("crypto_data.csv", parse_dates=["date"])

print(df.shape)
print(df.dtypes)
print(df["coin_id"].nunique(), "coins")
print(df["date"].min(), "→", df["date"].max())
print(df.isnull().sum())

(677378, 20)
date                  datetime64[us]
open                         float64
high                         float64
low                          float64
close                        float64
volume_usd                   float64
pair                             str
exchange                         str
coin_id                          str
symbol                           str
name                             str
market_cap_usd                 int64
cap_tier                         str
ath                          float64
atl                          float64
circulating_supply           float64
max_supply                   float64
change_24h_pct               float64
volume_24h_usd               float64
is_stablecoin                   bool
dtype: object
732 coins
1970-01-01 00:00:00 → 2026-04-17 00:00:00
date                       0
open                       0
high                       0
low                        0
close                      0
volume_usd                 0
pair   

In [3]:
summary = (
    df.groupby("coin_id")
    .agg(
        trading_days   = ("date", "count"),
        first_date     = ("date", "min"),
        last_date      = ("date", "max"),
        nan_close_pct  = ("close",      lambda x: x.isnull().mean() * 100),
        nan_volume_pct = ("volume_usd", lambda x: x.isnull().mean() * 100),
        median_volume  = ("volume_usd", "median"),
        is_stablecoin  = ("is_stablecoin", "first"),
        symbol         = ("symbol", "first"),
        name           = ("name",   "first"),
        cap_tier       = ("cap_tier", "first"),
    )
    .reset_index()
)

total_days = (df["date"].max() - df["date"].min()).days + 1
print(f"Calendar days in dataset: {total_days}")
print(summary.shape)
summary.head()

Calendar days in dataset: 20561
(732, 11)


,coin_id,trading_days,first_date,last_date,nan_close_pct,nan_volume_pct,median_volume,is_stablecoin,symbol,name,cap_tier
0,0x,1824,2021-04-20,2026-04-17,0.0,0.0,2.194449e+06,False,ZRX,0x Protocol,micro
1,1inch,1824,2021-04-20,2026-04-17,0.0,0.0,3.613424e+06,False,1INCH,1INCH,small
2,aave,1824,2021-04-20,2026-04-17,0.0,0.0,2.100431e+07,False,AAVE,Aave,mid
3,access-protocol,1158,2023-02-15,2026-04-17,0.0,0.0,1.013549e+04,False,ACS,Access Protocol,micro
4,achain,523,2024-11-11,2026-04-17,0.0,0.0,5.521125e+06,False,ACT,Achain,micro


In [4]:
MIN_DAYS = 400

coverage_mask = summary["trading_days"] >= MIN_DAYS
dropped_coverage = summary[~coverage_mask]

print(f"Dropped (< {MIN_DAYS} days): {len(dropped_coverage)}")
print(dropped_coverage[["coin_id", "symbol", "trading_days"]].to_string())

summary = summary[coverage_mask].copy()
print(f"Remaining: {len(summary)} coins")

Dropped (< 400 days): 217
                            coin_id     symbol  trading_days
7                           acurast        ACU            88
9                         adi-token        ADI           130
16                     agora-dollar       AUSD           100
17                   ai-rig-complex        ARC           366
22                            akedo        AKE           239
23                     alchemist-ai       ALCH           365
26                             aleo       ALEO            79
29                           allora       ALLO           158
36                            anoma        XAN           201
37                         anyspend        ANY           117
41                          apriori        APR           177
42                             apro         AT           142
53                          aster-2      ASTER           194
58                      aura-on-sol       AURA           226
66                          avantis       AVNT           21

In [5]:
# Flag 1: explicit is_stablecoin column
stablecoin_mask = summary["is_stablecoin"] == True
print(f"Flagged via is_stablecoin: {stablecoin_mask.sum()}")

# Flag 2: manual keyword catch — algorithmic stablecoins often slip through
stablecoin_keywords = [
    "usd", "usdt", "usdc", "dai", "busd", "tusd", "ust", "frax",
    "lusd", "susd", "gusd", "husd", "eur", "gbp", "stable"
]
keyword_mask = summary["symbol"].str.lower().str.contains(
    "|".join(stablecoin_keywords), na=False
)
print(f"Flagged via keyword: {keyword_mask.sum()}")
print("Keyword hits:", summary[keyword_mask & ~stablecoin_mask][["coin_id","symbol"]].values)

# Combine and drop
drop_stable = stablecoin_mask | keyword_mask
summary["drop_reason_stable"] = drop_stable
print(f"\nTotal stablecoins removed: {drop_stable.sum()}")
summary = summary[~drop_stable].copy()
print(f"Remaining: {len(summary)} coins")

Flagged via is_stablecoin: 12
Flagged via keyword: 18
Keyword hits: [['anchored-coins-eur' 'AEUR']
 ['eurite' 'EURI']
 ['global-dollar' 'USDG']
 ['ring-usd' 'USDR']
 ['schuman-europ' 'EUROP']
 ['stablr-euro' 'EURR']
 ['usds' 'USDS']]

Total stablecoins removed: 19
Remaining: 496 coins


In [6]:
# If any keyword hits are NOT stablecoins, whitelist them:
false_positives = ["USDX"]  # add any here after reviewing
fp_mask = summary["symbol"].isin(false_positives)
# (already excluded above, so this is just for documentation)

In [8]:
# Work on filtered df only
valid_coins = summary["coin_id"].tolist()
df = df[df["coin_id"].isin(valid_coins)].copy()
df = df.sort_values(["coin_id", "date"]).reset_index(drop=True)

def audit_gaps(group):
    coin = group.name  # coin_id comes from groupby key, not the column
    is_nan = group["close"].isnull()

    # No gaps at all — return empty early
    if is_nan.sum() == 0:
        return pd.DataFrame(columns=["gap_length", "gap_start", "coin_id"])

    run_id = (is_nan != is_nan.shift()).cumsum()

    runs = (
        group[is_nan]
        .groupby(run_id)
        .agg(
            gap_length=("date", "count"),
            gap_start=("date", "min")
        )
        .reset_index(drop=True)
    )
    runs["coin_id"] = coin
    return runs

gap_report = (
    df.groupby("coin_id", group_keys=False)
    .apply(audit_gaps)
    .reset_index(drop=True)
)

print("Gap distribution:")
print(gap_report["gap_length"].value_counts().sort_index() if len(gap_report) > 0 else "No gaps found")

bad_gaps = gap_report[gap_report["gap_length"] > 7]["coin_id"].unique()
print(f"\nCoins with gap > 7 days: {len(bad_gaps)}")
print(bad_gaps)

Gap distribution:
No gaps found

Coins with gap > 7 days: 0
[]


In [9]:
# Drop coins with gaps > 7 consecutive days
summary = summary[~summary["coin_id"].isin(bad_gaps)].copy()
df      = df[~df["coin_id"].isin(bad_gaps)].copy()
print(f"Dropped for long gaps: {len(bad_gaps)}")
print(f"Remaining: {summary['coin_id'].nunique()} coins")

# Forward fill gaps <= 7 days (already filtered out worse ones)
df["close"]      = df.groupby("coin_id")["close"].ffill()
df["volume_usd"] = df.groupby("coin_id")["volume_usd"].ffill()
print("Forward fill done. NaNs remaining:", df[["close","volume_usd"]].isnull().sum().sum())

Dropped for long gaps: 0
Remaining: 496 coins
Forward fill done. NaNs remaining: 0


In [10]:
# Compute log returns per coin first (needed for Amihud)
df["log_return"] = df.groupby("coin_id")["close"].transform(
    lambda x: np.log(x / x.shift(1))
)

df["amihud_daily"] = df["log_return"].abs() / df["volume_usd"].replace(0, np.nan)

amihud = (
    df.groupby("coin_id")["amihud_daily"]
    .mean()
    .rename("amihud_ratio")
    .reset_index()
)

summary = summary.merge(amihud, on="coin_id", how="left")

# Hard floor: median daily volume < $500K
summary["drop_low_volume"] = summary["median_volume"] < 500_000

# Top 20% most illiquid
illiq_threshold = summary["amihud_ratio"].quantile(0.80)
summary["drop_illiquid"] = summary["amihud_ratio"] > illiq_threshold

print(f"Amihud threshold (80th pct): {illiq_threshold:.6f}")
print(f"Drop — low volume (<$500K): {summary['drop_low_volume'].sum()}")
print(f"Drop — illiquid (top 20%):  {summary['drop_illiquid'].sum()}")
print(f"Drop — either:              {(summary['drop_low_volume'] | summary['drop_illiquid']).sum()}")

drop_liquidity = summary["drop_low_volume"] | summary["drop_illiquid"]
summary = summary[~drop_liquidity].copy()
df      = df[df["coin_id"].isin(summary["coin_id"])].copy()
print(f"\nWorking universe: {len(summary)} coins")

Amihud threshold (80th pct): 0.000000
Drop — low volume (<$500K): 140
Drop — illiquid (top 20%):  99
Drop — either:              140

Working universe: 356 coins


In [11]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings("ignore")

def test_stationarity(series):
    s = series.dropna()
    if len(s) < 50:
        return {"adf_stat": np.nan, "adf_p": np.nan,
                "kpss_stat": np.nan, "kpss_p": np.nan, "verdict": "insufficient_data"}

    adf_stat, adf_p, _, _, _, _ = adfuller(s, autolag="AIC")

    try:
        kpss_stat, kpss_p, _, _ = kpss(s, regression="c", nlags="auto")
    except Exception:
        kpss_stat, kpss_p = np.nan, np.nan

    # ADF: reject H0 → stationary (p < 0.05)
    # KPSS: reject H0 → non-stationary (p < 0.05)
    adf_stationary  = adf_p  < 0.05
    kpss_stationary = kpss_p >= 0.05

    if adf_stationary and kpss_stationary:
        verdict = "stationary"
    elif not adf_stationary and not kpss_stationary:
        verdict = "unit_root"
    else:
        verdict = "conflicting"  # → needs ARFIMA

    return {
        "adf_stat": round(adf_stat, 4),
        "adf_p":    round(adf_p, 4),
        "kpss_stat": round(kpss_stat, 4),
        "kpss_p":   round(kpss_p, 4),
        "verdict":  verdict,
    }

results = []
coins = df["coin_id"].unique()

for coin in coins:
    series = df[df["coin_id"] == coin]["log_return"]
    r = test_stationarity(series)
    r["coin_id"] = coin
    results.append(r)

stationarity_df = pd.DataFrame(results)
print(stationarity_df["verdict"].value_counts())

verdict
stationary     351
conflicting      5
Name: count, dtype: int64


In [15]:
# pip install fracdiff-modern
# ARFIMA for conflicting coins

from fracdiff.sklearn import FracdiffStat

conflicting_coins = stationarity_df[stationarity_df["verdict"] == "conflicting"]["coin_id"].tolist()
print(f"Conflicting coins needing ARFIMA: {len(conflicting_coins)}")

d_params = {}
fd = FracdiffStat(window=100, mode="valid")  # estimates minimum d for stationarity

for coin in conflicting_coins:
    series = df[df["coin_id"] == coin]["log_return"].dropna().values.reshape(-1, 1)
    try:
        fd.fit(series)
        d_val = round(fd.d_[0], 4)
    except Exception as e:
        d_val = np.nan
        print(f"  {coin}: failed — {e}")
    d_params[coin] = d_val
    print(f"  {coin}: d = {d_val}")

stationarity_df["d_param"] = stationarity_df["coin_id"].map(d_params).fillna(0.0)

# Decision:
# d ≈ 0     → stationary, use as-is
# d ≈ 1     → I(1), log returns already correct
# 0 < d < 0.5 → apply fractional differencing
stationarity_df["needs_fracdiff"] = (
    (stationarity_df["d_param"] > 0.05) &
    (stationarity_df["d_param"] < 0.5)
)
print(f"\nCoins needing fractional differencing: {stationarity_df['needs_fracdiff'].sum()}")

Conflicting coins needing ARFIMA: 5
  celestia: d = 0.0
  galatasaray-fan-token: d = 0.0
  internet-computer: d = 0.0
  project-galaxy: d = 0.0
  zcash: d = 0.0

Coins needing fractional differencing: 0


In [17]:
from statsmodels.stats.multitest import multipletests

def apply_bh(pvalues, alpha=0.05):
    """Apply Benjamini-Hochberg correction. Returns boolean mask of significant results."""
    clean = np.where(np.isnan(pvalues), 1.0, pvalues)
    reject, pvals_corrected, _, _ = multipletests(clean, alpha=alpha, method="fdr_bh")
    return reject, pvals_corrected

# Apply to ADF p-values as first use
reject, adf_p_corrected = apply_bh(stationarity_df["adf_p"].values)
stationarity_df["adf_p_bh"] = adf_p_corrected

print("BH-FDR correction set up. Ready to apply in all downstream phases.")
print(f"ADF significant after BH: {reject.sum()} / {len(reject)}")

# Merge stationarity results into summary
summary = summary.merge(
    stationarity_df[["coin_id", "verdict", "d_param", "needs_fracdiff", "adf_p_bh"]],
    on="coin_id", how="left"
)

# Save
summary.to_csv("01_universe_filtered.csv", index=False)

# Clean returns matrix for Phase 2
returns_matrix = (
    df[df["coin_id"].isin(summary["coin_id"])]
    .pivot(index="date", columns="coin_id", values="log_return")
    .sort_index()
)
returns_matrix.to_parquet("02_returns_matrix.parquet")

print(f"\nPhase 1 complete.")
print(f"  Final universe:  {len(summary)} coins")
print(f"  Returns matrix:  {returns_matrix.shape}  (days × coins)")
print(f"  Stationary:      {(summary['verdict'] == 'stationary').sum()}")
print(f"  Conflicting:     {(summary['verdict'] == 'conflicting').sum()}")
print(f"  Needs fracdiff:  {summary['needs_fracdiff'].sum()}")
summary[["coin_id","symbol","trading_days","amihud_ratio","median_volume","verdict","d_param"]].head(10)

BH-FDR correction set up. Ready to apply in all downstream phases.
ADF significant after BH: 356 / 356

Phase 1 complete.
  Final universe:  356 coins
  Returns matrix:  (1824, 356)  (days × coins)
  Stationary:      351
  Conflicting:     5
  Needs fracdiff:  0


,coin_id,symbol,trading_days,amihud_ratio,median_volume,verdict,d_param
0,0x,ZRX,1824,2.060357e-08,2.194449e+06,stationary,0.0
1,1inch,1INCH,1824,1.145222e-08,3.613424e+06,stationary,0.0
2,aave,AAVE,1824,2.191446e-09,2.100431e+07,stationary,0.0
3,achain,ACT,523,1.121609e-08,5.521125e+06,stationary,0.0
4,across-protocol,ACX,498,6.201279e-08,1.028876e+06,stationary,0.0
5,act-i-the-ai-prophecy,ACT,523,1.121609e-08,5.521125e+06,stationary,0.0
6,adex,ADX,1632,2.867357e-08,1.142882e+06,stationary,0.0
7,adventure-gold,AGLD,1656,2.987033e-08,1.628248e+06,stationary,0.0
8,aelf,ELF,1317,2.323998e-08,1.441143e+06,stationary,0.0
9,aergo,AERGO,694,4.487836e-08,8.396699e+05,stationary,0.0
